# Notebook de replicación
## Informe técnico OMSCGR — *¿Qué explica la mejora en la percepción de inseguridad? Evidencia del Centro Histórico de Quito, 2024-2026*

**Autor:** Juan Pablo Jaramillo-Ramón · Observatorio Metropolitano de Seguridad Ciudadana y Gestión de Riesgos (OMSCGR)
**Datos:** Estudio de Victimización y Percepción de Seguridad Ciudadana (EVPSC), Centro Histórico de Quito — olas 2024, 2025 y 2026.

Este notebook reproduce todas las cifras del informe a partir de la base armonizada `CHQ_percepcion_2024_2026_replicacion.csv`:

| Sección | Contenido |
|---|---|
| 2 | Cuadro 1 — percepción de inseguridad y victimización por actor, 2024 → 2025 → 2026 |
| 3 | Muestra analítica del modelo (2024 y 2026) y cifras de la introducción |
| 4 | Cuadro 2 — efectos marginales promedio del modelo logit agrupado |
| 5 | Cuadro 3 — descomposición Oaxaca-Blinder/Fairlie de la brecha 2024-2026 y cifras de la sección 3.3 |
| 6 | Extensión — intervalos de confianza bootstrap de la descomposición (no incluidos en el informe) |
| 7 | Verificación automática contra los valores impresos en el informe |

**Cómo correrlo en Google Colab:** *Archivo → Subir notebook*, luego *Entorno de ejecución → Ejecutar todas*. Cuando lo pida, suba `CHQ_percepcion_2024_2026_replicacion.csv` (o pegue en `DATA_URL` un enlace público al CSV). Tiempo de ejecución: menos de un minuto.

In [1]:
import os, platform, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | statsmodels {statsmodels.__version__}")

def fmt(x, d=1):
    # Formato del informe: coma decimal
    return f"{x:.{d}f}".replace('.', ',')

Python 3.12.3 | pandas 3.0.2 | numpy 2.4.4 | statsmodels 0.15.0


## 1. Carga de la base
Una fila por persona encuestada (N = 2.448: 818 en 2024, 775 en 2025 y 855 en 2026). La base se construyó a partir de las bases crudas del OMSCGR con `00_construir_base.py` (incluido en el paquete). Las definiciones de variables están en `libro_de_codigos.csv`.

In [2]:
DATA_FILE = 'CHQ_percepcion_2024_2026_replicacion.csv'
DATA_URL = ''   # opcional: enlace público (GitHub/OSF) al CSV

if DATA_URL:
    data = pd.read_csv(DATA_URL)
else:
    if not os.path.exists(DATA_FILE):
        try:
            from google.colab import files
            print(f"Suba el archivo {DATA_FILE}")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"No se encontró {DATA_FILE} en {os.getcwd()}")
    data = pd.read_csv(DATA_FILE)

print(data.shape)
print(data.groupby(['anio', 'tipo_actor']).size().unstack())
data.head()

(2448, 19)
tipo_actor  autonomo  flotante  locales
anio                                   
2024              76       392      350
2025              79       392      304
2026              96       391      368


,id,anio,tipo_actor,inseguro,victima,vio_ffaa,vio_policia,vio_cacmq,cc_peleas,cc_situacion_calle,cc_alcohol,cc_drogas,cc_danio_propiedad,cc_fauna_urbana,cc_basura,cc_ruido,cc_aglomeracion,ferias,hombre
0,CHQ24_0001,2024,locales,0,0,0,1,1,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1
1,CHQ24_0002,2024,locales,1,0,1,1,1,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1
2,CHQ24_0003,2024,locales,1,0,1,1,1,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0
3,CHQ24_0004,2024,locales,1,1,1,1,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
4,CHQ24_0005,2024,locales,1,0,1,1,1,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1


## 2. Cuadro 1 — Percepción de inseguridad y victimización por actor, 2024-2026
Porcentajes sobre todas las personas encuestadas de cada ola y actor (la victimización excluye respuestas perdidas). Incluye la ola 2025, que no entra en el modelo.

In [3]:
ACTORES = {'flotante': 'Población flotante', 'locales': 'Locales comerciales', 'autonomo': 'Comercio autónomo'}
c1 = (data.groupby(['tipo_actor', 'anio'])[['inseguro', 'victima']].mean() * 100).round(1)
filas = []
for k, lab in ACTORES.items():
    p = [c1.loc[(k, y), 'inseguro'] for y in (2024, 2025, 2026)]
    v = [c1.loc[(k, y), 'victima'] for y in (2024, 2025, 2026)]
    filas.append([lab, ' → '.join(fmt(x) + '%' for x in p), ' → '.join(fmt(x) + '%' for x in v)])
cuadro1 = pd.DataFrame(filas, columns=['Actor', 'Percepción insegura 2024 → 2025 → 2026', 'Victimización 2024 → 2025 → 2026'])

serie = (data.groupby('anio')[['inseguro', 'victima']].mean() * 100).round(1)
print('Promedio de los tres actores (ponderado por tamaño de muestra):')
print(serie.rename(columns={'inseguro': '% inseguro', 'victima': '% víctima'}))
cuadro1

Promedio de los tres actores (ponderado por tamaño de muestra):
      % inseguro  % víctima
anio                       
2024        66.4       33.4
2025        56.1       27.5
2026        53.7       31.8


,Actor,Percepción insegura 2024 → 2025 → 2026,Victimización 2024 → 2025 → 2026
0,Población flotante,"59,7% → 51,5% → 53,7%","15,1% → 15,8% → 20,7%"
1,Locales comerciales,"72,6% → 62,2% → 54,9%","50,0% → 39,1% → 39,9%"
2,Comercio autónomo,"72,4% → 55,7% → 49,0%","51,3% → 40,5% → 45,8%"


## 3. Muestra analítica del modelo (2024 y 2026)
El modelo usa solo las olas 2024 y 2026 y elimina observaciones con valores perdidos en alguna covariable: **1.607 personas** (766 en 2024; 841 en 2026). La categoría de referencia del tipo de actor es el **comercio autónomo**.

In [4]:
CC = ['cc_peleas', 'cc_situacion_calle', 'cc_alcohol', 'cc_drogas', 'cc_danio_propiedad',
      'cc_fauna_urbana', 'cc_basura', 'cc_ruido', 'cc_aglomeracion']
INST = ['vio_cacmq', 'vio_policia', 'vio_ffaa']

df = data[data['anio'].isin([2024, 2026])].dropna().copy()
df['anio2026'] = (df['anio'] == 2026).astype(int)
df['actor_flotante'] = (df['tipo_actor'] == 'flotante').astype(int)
df['actor_locales'] = (df['tipo_actor'] == 'locales').astype(int)
COV = ['anio2026', 'hombre', 'victima', 'ferias'] + CC + INST + ['actor_flotante', 'actor_locales']

print('N analítico =', len(df), '|', df.groupby('anio').size().to_dict())
medias = (df.groupby('anio')[['inseguro', 'victima']].mean() * 100).round(1)
print('\nCifras de la introducción (muestra analítica):'); print(medias)

N analítico = 1607 | {2024: 766, 2026: 841}

Cifras de la introducción (muestra analítica):
      inseguro  victima
anio                   
2024      65.9     32.4
2026      53.5     31.9


In [5]:
ETIQ = {'anio2026': 'Año 2026 (vs. 2024, a igual composición)', 'hombre': 'Ser hombre (vs. mujer)',
        'victima': 'Haber sido víctima de un delito', 'ferias': 'Haber presenciado ferias gastronómicas/culturales',
        'cc_peleas': 'Peleas o riñas en el espacio público', 'cc_situacion_calle': 'Personas en situación de calle',
        'cc_alcohol': 'Consumo de alcohol en el espacio público', 'cc_drogas': 'Consumo de drogas en el espacio público',
        'cc_danio_propiedad': 'Daño a la propiedad pública/privada', 'cc_fauna_urbana': 'Fauna urbana (animales sin hogar)',
        'cc_basura': 'Basura en los espacios públicos', 'cc_ruido': 'Ruidos molestos', 'cc_aglomeracion': 'Aglomeración de personas',
        'actor_flotante': 'Ser población flotante (vs. comercio autónomo)', 'actor_locales': 'Ser local comercial (vs. comercio autónomo)',
        'vio_cacmq': 'Vio trabajar al CACMQ', 'vio_policia': 'Vio trabajar a la Policía Nacional', 'vio_ffaa': 'Vio trabajar a las Fuerzas Armadas'}

def logit(d, cov):
    X = sm.add_constant(d[cov].astype(float), has_constant='add')
    return sm.Logit(d['inseguro'].astype(float), X).fit(disp=0, cov_type='HC1')   # errores robustos HC1

def efectos_marginales(m):
    sf = m.get_margeff(at='overall').summary_frame()   # efectos marginales promedio (AME)
    return sf.rename(columns={'dy/dx': 'AME', 'Std. Err.': 'EE', 'Pr(>|z|)': 'p'})[['AME', 'EE', 'p']]

## 4. Cuadro 2 — Efectos marginales promedio del modelo logístico agrupado (2024 + 2026)

In [6]:
m = logit(df, COV)
ame = efectos_marginales(m).sort_values('AME', key=np.abs, ascending=False)
estrellas = lambda p: '**' if p < 0.01 else ('*' if p < 0.05 else '')
cuadro2 = pd.DataFrame({
    'Variable': [ETIQ[v] for v in ame.index],
    'Efecto marginal': [f"{'+' if a >= 0 else ''}{fmt(a*100)} p.p.{estrellas(p)}" for a, p in zip(ame['AME'], ame['p'])],
    'Valor p': [fmt(p, 3) for p in ame['p']]})
print(f"N = {int(m.nobs)} | pseudo-R² = {m.prsquared:.3f}   (* p<0,05; ** p<0,01)")
cuadro2

N = 1607 | pseudo-R² = 0.143   (* p<0,05; ** p<0,01)


,Variable,Efecto marginal,Valor p
0,Personas en situación de calle,"+21,3 p.p.**","0,000"
1,Haber sido víctima de un delito,"+18,5 p.p.**","0,000"
2,Ser hombre (vs. mujer),"-12,5 p.p.**","0,000"
3,Vio trabajar a la Policía Nacional,"-9,7 p.p.**","0,002"
4,"Año 2026 (vs. 2024, a igual composición)","-9,3 p.p.**","0,000"
5,Consumo de drogas en el espacio público,"+9,0 p.p.**","0,001"
6,Ser población flotante (vs. comercio autónomo),"+8,7 p.p.*","0,035"
7,Peleas o riñas en el espacio público,"+8,3 p.p.**","0,002"
8,Consumo de alcohol en el espacio público,"+7,8 p.p.**","0,008"
9,Vio trabajar al CACMQ,"-7,7 p.p.*","0,039"


## 5. Cuadro 3 — Descomposición Oaxaca-Blinder/Fairlie de la brecha 2024-2026
Se estiman dos logit separados (2024 y 2026) con las mismas covariables, sin el indicador de año. Sea $\bar P(\hat\beta_t, X_s)$ la probabilidad media predicha con coeficientes de la ola $t$ y covariables de la ola $s$:

* **Referencia 2024:** explicado $=\bar P(\hat\beta_{24},X_{24})-\bar P(\hat\beta_{24},X_{26})$; no explicado $=\bar P(\hat\beta_{24},X_{26})-\bar P(\hat\beta_{26},X_{26})$.
* **Referencia 2026:** explicado $=\bar P(\hat\beta_{26},X_{24})-\bar P(\hat\beta_{26},X_{26})$; no explicado $=\bar P(\hat\beta_{24},X_{24})-\bar P(\hat\beta_{26},X_{24})$.

La contribución de cada variable usa la aproximación lineal de Yun (2004), $\hat\beta_k(\bar x_{k,26}-\bar x_{k,24})\,\Lambda'(\bar x\hat\beta)$ en las medias conjuntas, con el signo invertido: **valores positivos contribuyen a la mejora** (menos inseguridad). La aproximación no suma exactamente el total no lineal.

In [7]:
DCOV = [c for c in COV if c != 'anio2026']

def descomponer(d24, d26):
    X24 = sm.add_constant(d24[DCOV].astype(float), has_constant='add')
    X26 = sm.add_constant(d26[DCOV].astype(float), has_constant='add')
    m24 = sm.Logit(d24['inseguro'].astype(float), X24).fit(disp=0, cov_type='HC1')
    m26 = sm.Logit(d26['inseguro'].astype(float), X26).fit(disp=0, cov_type='HC1')
    P = lambda mod, X: mod.predict(X).mean()
    p24, p26 = P(m24, X24), P(m26, X26)
    r = {'p24': p24, 'p26': p26, 'brecha': p24 - p26,
         'expl_24': p24 - P(m24, X26), 'noexpl_24': P(m24, X26) - p26,
         'expl_26': P(m26, X24) - p26, 'noexpl_26': p24 - P(m26, X24), 'm24': m24, 'm26': m26}
    xbar = pd.concat([X24, X26]).mean()
    for ref, mod in (('24', m24), ('26', m26)):
        pb = 1 / (1 + np.exp(-(xbar * mod.params).sum()))
        r[f'detalle_{ref}'] = {k: -mod.params[k] * (X26[k].mean() - X24[k].mean()) * pb * (1 - pb) * 100 for k in DCOV}
    return r

d24, d26 = df[df['anio'] == 2024], df[df['anio'] == 2026]
D = descomponer(d24, d26)
g = D['brecha']
cuadro3 = pd.DataFrame({
    'Referencia: coeficientes 2024': [f"{fmt(D['expl_24']*100)} p.p. ({fmt(D['expl_24']/g*100)}%)", f"{fmt(D['noexpl_24']*100)} p.p. ({fmt(D['noexpl_24']/g*100)}%)"],
    'Referencia: coeficientes 2026': [f"{fmt(D['expl_26']*100)} p.p. ({fmt(D['expl_26']/g*100)}%)", f"{fmt(D['noexpl_26']*100)} p.p. ({fmt(D['noexpl_26']/g*100)}%)"]},
    index=['Explicado por composición', 'No explicado (cambio de relación)'])
print(f"Probabilidad media de sentirse inseguro: 2024 = {fmt(D['p24']*100)}% | 2026 = {fmt(D['p26']*100)}% | brecha = {fmt(g*100)} p.p.")
cuadro3

Probabilidad media de sentirse inseguro: 2024 = 65,9% | 2026 = 53,5% | brecha = 12,4 p.p.


,Referencia: coeficientes 2024,Referencia: coeficientes 2026
Explicado por composición,"3,4 p.p. (27,6%)","2,8 p.p. (22,4%)"
No explicado (cambio de relación),"9,0 p.p. (72,4%)","9,6 p.p. (77,6%)"


In [8]:
# Contribución de cada variable (p.p.; + = contribuye a la mejora) y orden con cada referencia
det = pd.DataFrame({'Ref. 2024': D['detalle_24'], 'Ref. 2026': D['detalle_26']})
det['Orden ref. 2024'] = det['Ref. 2024'].rank(ascending=False).astype(int)
det['Orden ref. 2026'] = det['Ref. 2026'].rank(ascending=False).astype(int)
det.index = [ETIQ[k] for k in det.index]
display(det.sort_values('Ref. 2024', ascending=False).round(2))

# Cifras citadas en la sección 3.3 del informe
cit = ['cc_situacion_calle', 'vio_policia', 'cc_peleas', 'cc_aglomeracion', 'vio_ffaa']
print((df.groupby('anio')[cit].mean().T * 100).round(1).rename(index=ETIQ))
print(f"\nCoeficiente logit de 'Vio trabajar a las FFAA': 2024 = {D['m24'].params['vio_ffaa']:+.2f} (p = {D['m24'].pvalues['vio_ffaa']:.3f}) | "
      f"2026 = {D['m26'].params['vio_ffaa']:+.2f} (p = {D['m26'].pvalues['vio_ffaa']:.3f})")

,Ref. 2024,Ref. 2026,Orden ref. 2024,Orden ref. 2026
Personas en situación de calle,1.52,2.03,1,1
Vio trabajar a las Fuerzas Armadas,0.91,-1.76,2,17
Basura en los espacios públicos,0.88,0.35,3,7
Consumo de drogas en el espacio público,0.76,1.19,4,2
Vio trabajar a la Policía Nacional,0.51,0.94,5,4
Consumo de alcohol en el espacio público,0.50,0.43,6,6
Fauna urbana (animales sin hogar),0.44,1.06,7,3
Daño a la propiedad pública/privada,0.30,-0.22,8,13
Ser población flotante (vs. comercio autónomo),0.26,0.43,9,5
Haber sido víctima de un delito,0.10,0.13,10,8


anio                                  2024  2026
Personas en situación de calle        93.3  86.2
Vio trabajar a la Policía Nacional    74.9  81.6
Peleas o riñas en el espacio público  69.3  73.5
Aglomeración de personas              66.3  56.2
Vio trabajar a las Fuerzas Armadas    53.9  74.2

Coeficiente logit de 'Vio trabajar a las FFAA': 2024 = -0.20 (p = 0.276) | 2026 = +0.35 (p = 0.073)


## 6. Extensión — intervalos de confianza bootstrap de la descomposición
**No incluida en el informe.** Bootstrap no paramétrico que remuestrea personas dentro de cada ola (B = 300, semilla fija). Muestra cuán precisa es la partición entre componente explicado y no explicado.

In [9]:
rng = np.random.default_rng(20260924)
B, boot = 300, []
for b in range(B):
    s24 = d24.sample(len(d24), replace=True, random_state=rng.integers(1e9))
    s26 = d26.sample(len(d26), replace=True, random_state=rng.integers(1e9))
    try:
        r = descomponer(s24, s26)
        boot.append([r['brecha']*100, r['expl_24']/r['brecha']*100, r['expl_26']/r['brecha']*100,
                     r['detalle_24']['vio_ffaa'], r['detalle_26']['vio_ffaa'],
                     r['detalle_24']['vio_policia'], r['detalle_26']['vio_policia'],
                     r['detalle_24']['cc_situacion_calle'], r['detalle_26']['cc_situacion_calle']])
    except Exception:
        pass
boot = pd.DataFrame(boot, columns=['Brecha (p.p.)', '% explicado, ref. 2024', '% explicado, ref. 2026',
                                   'Contrib. FFAA, ref. 2024', 'Contrib. FFAA, ref. 2026',
                                   'Contrib. Policía, ref. 2024', 'Contrib. Policía, ref. 2026',
                                   'Contrib. situación de calle, ref. 2024', 'Contrib. situación de calle, ref. 2026'])
print(f"Réplicas exitosas: {len(boot)}/{B}")
boot.quantile([0.025, 0.5, 0.975]).T.round(2).rename(columns={0.025: 'IC 2,5%', 0.5: 'Mediana', 0.975: 'IC 97,5%'})

Réplicas exitosas: 300/300


,"IC 2,5%",Mediana,"IC 97,5%"
Brecha (p.p.),7.64,12.57,17.86
"% explicado, ref. 2024",1.20,26.06,53.45
"% explicado, ref. 2026",-5.82,21.50,53.31
"Contrib. FFAA, ref. 2024",-0.52,0.88,2.61
"Contrib. FFAA, ref. 2026",-4.20,-1.71,0.35
"Contrib. Policía, ref. 2024",-0.10,0.47,1.37
"Contrib. Policía, ref. 2026",0.13,0.88,2.13
"Contrib. situación de calle, ref. 2024",0.44,1.43,3.23
"Contrib. situación de calle, ref. 2026",1.03,1.99,3.60


## 7. Verificación contra el informe
Compara cada cifra impresa en el informe con el valor replicado. La tolerancia es media unidad del último decimal impreso. El último bloque revisa afirmaciones del texto redactadas con cifras aproximadas.

In [10]:
pp = lambda v: ame.loc[v, 'AME'] * 100
chk = []
def num(fuente, nombre, informe, replicado):
    tol = 0.5 * 10 ** -len(repr(abs(informe)).split('.')[1]) + 1e-9
    chk.append([fuente, nombre, informe, round(float(replicado), 3), 'OK' if abs(informe - replicado) <= tol else 'DIFERENCIA'])

# Cuadro 1
REP = {('flotante','inseguro'): (59.7, 51.5, 53.7), ('locales','inseguro'): (72.6, 62.2, 54.9), ('autonomo','inseguro'): (72.4, 55.7, 49.0),
       ('flotante','victima'): (15.1, 15.8, 20.7), ('locales','victima'): (50.0, 39.1, 39.9), ('autonomo','victima'): (51.3, 40.5, 45.8)}
for (a, v), vals in REP.items():
    for y, x in zip((2024, 2025, 2026), vals):
        num('Cuadro 1', f"{v} {a} {y}", x, c1.loc[(a, y), v])
# Introducción
num('Introducción', '% inseguro 2024 (muestra analítica)', 65.9, medias.loc[2024, 'inseguro'])
num('Introducción', '% inseguro 2026 (muestra analítica)', 53.5, medias.loc[2026, 'inseguro'])
num('Introducción', 'brecha (p.p.)', 12.4, D['brecha'] * 100)
num('Introducción', '% víctima 2024 (muestra analítica)', 32.4, medias.loc[2024, 'victima'])
num('Introducción', '% víctima 2026 (muestra analítica)', 31.9, medias.loc[2026, 'victima'])
for y, x in zip((2024, 2025, 2026), (33.4, 27.5, 31.8)):
    num('Introducción', f'% víctima, tres actores {y}', x, serie.loc[y, 'victima'])
# Cuadro 2
C2 = {'cc_situacion_calle': 21.3, 'victima': 18.5, 'hombre': -12.5, 'vio_policia': -9.7, 'anio2026': -9.3, 'cc_drogas': 9.0,
      'actor_flotante': 8.7, 'cc_peleas': 8.3, 'cc_alcohol': 7.8, 'vio_cacmq': -7.7, 'cc_aglomeracion': -6.0, 'actor_locales': 5.2,
      'cc_ruido': 4.6, 'cc_fauna_urbana': 3.9, 'ferias': -2.7, 'cc_basura': 2.3, 'vio_ffaa': 2.0, 'cc_danio_propiedad': 0.4}
for v, x in C2.items():
    num('Cuadro 2', f"AME {ETIQ[v]}", x, pp(v))
num('Cuadro 2', 'p Policía', 0.002, ame.loc['vio_policia', 'p']); num('Cuadro 2', 'p CACMQ', 0.039, ame.loc['vio_cacmq', 'p'])
num('Cuadro 2', 'p FFAA', 0.441, ame.loc['vio_ffaa', 'p']); num('Cuadro 2', 'p aglomeración', 0.021, ame.loc['cc_aglomeracion', 'p'])
# Cuadro 3
for nombre, x, r in [('explicado ref. 2024 (p.p.)', 3.4, D['expl_24']*100), ('% explicado ref. 2024', 27.6, D['expl_24']/g*100),
                     ('explicado ref. 2026 (p.p.)', 2.8, D['expl_26']*100), ('% explicado ref. 2026', 22.4, D['expl_26']/g*100),
                     ('no explicado ref. 2024 (p.p.)', 9.0, D['noexpl_24']*100), ('% no explicado ref. 2024', 72.4, D['noexpl_24']/g*100),
                     ('no explicado ref. 2026 (p.p.)', 9.6, D['noexpl_26']*100), ('% no explicado ref. 2026', 77.6, D['noexpl_26']/g*100)]:
    num('Cuadro 3', nombre, x, r)
# Sección 3.3
M = df.groupby('anio')[cit].mean() * 100
for v, a, b in [('cc_situacion_calle', 93.3, 86.2), ('vio_policia', 74.9, 81.6), ('cc_peleas', 69.3, 73.5), ('cc_aglomeracion', 66.3, 56.2), ('vio_ffaa', 53.9, 74.2)]:
    num('Sección 3.3', f"{ETIQ[v]} 2024", a, M.loc[2024, v]); num('Sección 3.3', f"{ETIQ[v]} 2026", b, M.loc[2026, v])
num('Sección 3.3', 'β FFAA 2024', -0.20, D['m24'].params['vio_ffaa']); num('Sección 3.3', 'β FFAA 2026', 0.35, D['m26'].params['vio_ffaa'])

tabla = pd.DataFrame(chk, columns=['Fuente', 'Cifra', 'Informe', 'Replicado', 'Estado'])
print(f"Cifras numéricas replicadas: {(tabla['Estado']=='OK').sum()}/{len(tabla)}")
display(tabla)

# Afirmaciones del texto redactadas con cifras aproximadas
lo, hi = sorted([D['noexpl_24']/g*100, D['noexpl_26']/g*100])
elo, ehi = sorted([D['expl_24']*100, D['expl_26']*100])
mej_loc = c1.loc[('locales', 2024), 'inseguro'] - c1.loc[('locales', 2026), 'inseguro']
mej_aut = c1.loc[('autonomo', 2024), 'inseguro'] - c1.loc[('autonomo', 2026), 'inseguro']
ratio = M.loc[2026, 'vio_ffaa'] / M.loc[2024, 'vio_ffaa']
texto = pd.DataFrame([
    ['Resumen', '"entre 69% y 82% de la mejora no se explica"', f"{fmt(lo)}% – {fmt(hi)}%"],
    ['Sección 3.3', '"entre 2,3 y 3,4 puntos porcentuales" explicados', f"{fmt(elo)} – {fmt(ehi)} p.p."],
    ['Sección 3.1', '"cerca de 20 y 23 puntos" de mejora (locales y autónomo)', f"{fmt(mej_loc)} y {fmt(mej_aut)} p.p."],
    ['Sección 3.3', 'presencia de FFAA "casi se duplicó"', f"×{fmt(ratio, 2)} (+{fmt(M.loc[2026,'vio_ffaa']-M.loc[2024,'vio_ffaa'])} p.p.)"],
    ['Sección 3.3', 'Policía es el "tercer mayor contribuyente"',
     f"puesto {det.loc[ETIQ['vio_policia'], 'Orden ref. 2024']} con ref. 2024; puesto {det.loc[ETIQ['vio_policia'], 'Orden ref. 2026']} con ref. 2026"]],
    columns=['Fuente', 'Texto del informe', 'Valor replicado'])
texto

Cifras numéricas replicadas: 68/68


,Fuente,Cifra,Informe,Replicado,Estado
0,Cuadro 1,inseguro flotante 2024,59.700,59.700,OK
1,Cuadro 1,inseguro flotante 2025,51.500,51.500,OK
2,Cuadro 1,inseguro flotante 2026,53.700,53.700,OK
3,Cuadro 1,inseguro locales 2024,72.600,72.600,OK
4,Cuadro 1,inseguro locales 2025,62.200,62.200,OK
5,Cuadro 1,inseguro locales 2026,54.900,54.900,OK
6,Cuadro 1,inseguro autonomo 2024,72.400,72.400,OK
7,Cuadro 1,inseguro autonomo 2025,55.700,55.700,OK
8,Cuadro 1,inseguro autonomo 2026,49.000,49.000,OK
9,Cuadro 1,victima flotante 2024,15.100,15.100,OK


,Fuente,Texto del informe,Valor replicado
0,Resumen,"""entre 69% y 82% de la mejora no se explica""","72,4% – 77,6%"
1,Sección 3.3,"""entre 2,3 y 3,4 puntos porcentuales"" explicados","2,8 – 3,4 p.p."
2,Sección 3.1,"""cerca de 20 y 23 puntos"" de mejora (locales y...","17,7 y 23,4 p.p."
3,Sección 3.3,"presencia de FFAA ""casi se duplicó""","×1,38 (+20,3 p.p.)"
4,Sección 3.3,"Policía es el ""tercer mayor contribuyente""",puesto 5 con ref. 2024; puesto 4 con ref. 2026


---
### Notas
* **Datos crudos.** Las bases EVPSC son propiedad del OMSCGR (Municipio del Distrito Metropolitano de Quito). `00_construir_base.py` documenta cada decisión de armonización: respuestas en texto plano en 2024 frente a respuestas con prefijo numérico en 2025-2026; exclusión de la ola fuera de ciclo de febrero de 2025; identificación de las instituciones por nombre en el bloque SSM; exclusión de 31 filas vacías en la base de comercio autónomo.
* **Inferencia.** Cortes transversales repetidos: las estimaciones son asociaciones, no efectos causales. Errores estándar robustos (HC1) en todos los modelos.